# YOLO 객체 탐지: 선택 실행

각 Notebook은 독립 실행합니다. 기본값 DEMO=True는 합성 연습 데이터입니다. 실제 데이터는 설정 셀에서 DEMO=False와 경로·열·문제 유형을 지정하세요. 앞의 Notebook 실행이나 개인 모듈 설치가 필요하지 않습니다.

시간 예산은 모델 후보를 시작하기 전에 확인하는 소프트 제한입니다. 진행 중인 fit을 강제 중단하지 않습니다. 대회 지문과 공식 제출 규격을 우선합니다.

## 설정

이 경로는 torch·ultralytics와 실제 탐지 데이터가 필요합니다. CPU에서는 오래 걸릴 수 있습니다. 기본 검증 환경에서는 의존성과 가중치가 없어 학습을 실행하지 않았습니다.

In [ ]:
RUN_YOLO=False # 패키지/로컬 가중치/실제 데이터를 준비한 뒤 True
DATA_YAML='data/detection.yaml' # Ultralytics 형식: path, train, val, names
LOCAL_WEIGHTS='weights/model.pt' # 사용이 허용된 로컬 탐지 가중치
TEST_SOURCE='data/test_images'
EPOCHS=3;IMAGE_SIZE=320;BATCH_SIZE=4
OUTPUT_DIR='outputs/yolo'


## 학습·검증·추론

data.yaml의 train/val은 촬영 세션 단위로 분리하고 YOLO normalized xywh 라벨을 준비합니다. 모델 이름 자동 다운로드 대신 허용된 로컬 파일만 지정합니다. 출력 long CSV는 공식 제출 규격에 맞게 변환하세요.

In [ ]:
from pathlib import Path
import pandas as pd
if RUN_YOLO:
    if not Path(DATA_YAML).is_file() or not Path(LOCAL_WEIGHTS).is_file() or not Path(TEST_SOURCE).exists():
        raise FileNotFoundError('로컬 data yaml, weights, test source를 준비하세요.')
    from ultralytics import YOLO
    model=YOLO(LOCAL_WEIGHTS)
    model.train(data=DATA_YAML,epochs=EPOCHS,imgsz=IMAGE_SIZE,batch=BATCH_SIZE,device='cpu',workers=0,seed=42,project=OUTPUT_DIR,name='train',exist_ok=True)
    best=Path(model.trainer.best)
    if not best.is_file(): raise RuntimeError('best checkpoint 생성 실패')
    final_model=YOLO(str(best));validation=final_model.val(data=DATA_YAML,device='cpu',workers=0)
    print(validation.results_dict)
    rows=[];counts=[]
    for result in final_model.predict(source=TEST_SOURCE,device='cpu',stream=True):
        counts.append({'path':result.path,'detection_count':len(result.boxes)})
        for xyxy,score,cls in zip(result.boxes.xyxy.cpu().numpy(),result.boxes.conf.cpu().numpy(),result.boxes.cls.cpu().numpy()):
            rows.append({'path':result.path,'class_id':int(cls),'confidence':float(score),**dict(zip(['xmin','ymin','xmax','ymax'],map(float,xyxy)))})
    out=Path(OUTPUT_DIR);out.mkdir(parents=True,exist_ok=True)
    pd.DataFrame(rows,columns=['path','class_id','confidence','xmin','ymin','xmax','ymax']).to_csv(out/'detections.csv',index=False)
    pd.DataFrame(counts).to_csv(out/'counts.csv',index=False)
else:
    print('선택 경로 비활성: 실제 데이터·허용 가중치·ultralytics가 있을 때만 RUN_YOLO=True로 실행합니다.')


## 공식 문서

[Ultralytics train](https://docs.ultralytics.com/modes/train/) · 설치는 별도 환경에서 requirements-optional.txt를 사용하고 대회 허용 규칙을 확인하세요.